# CV R² comparison: standard vs session-shift aPRF

Does allowing the preferred value (mode) to shift between sessions improve
cross-validated prediction? The session-shift model has one extra free
parameter per voxel (mode_2), so it should only win if the tuning genuinely
changes between conditions.

We compare three models:
1. **Null** — predict training mean (no stimulus dependence)
2. **Standard** — fixed mode across sessions
3. **Session-shift** — mode shifts freely per session

Voxels are filtered to those where **at least one** encoding model has CV R² > 0.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn.maskers import NiftiMasker
from nilearn import image as nli

from abstract_values.utils.data import Subject, BIDS_FOLDER

sns.set_theme(style='ticks', font_scale=1.1)

In [ ]:
SMOOTHED = False
smooth_label = '_smoothed' if SMOOTHED else ''

ROIS = [
    ('NPCr', None),
    ('BensonV1', 'LR'),
]

# Auto-discover subjects that have BOTH standard and shift CV R²
std_cv_dir  = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf.cv'
shift_cv_dir = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf-shift.cv'

SUBJECTS = []
for d in sorted(std_cv_dir.iterdir()) if std_cv_dir.exists() else []:
    if not d.is_dir() or not d.name.startswith('sub-'):
        continue
    sub_id = d.name.removeprefix('sub-')
    std_fn = d / 'func' / f'{d.name}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz'
    shift_fn = shift_cv_dir / d.name / 'func' / f'{d.name}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz'
    if std_fn.exists() and shift_fn.exists():
        SUBJECTS.append(sub_id)

print(f'Subjects with both standard and shift CV R²: {SUBJECTS}')

In [ ]:
from braincoder.utils import get_rsq

data = {}

for sub_id in SUBJECTS:
    sub = Subject(sub_id, bids_folder=BIDS_FOLDER)

    std_fn = (std_cv_dir / f'sub-{sub_id}' / 'func' /
              f'sub-{sub_id}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz')
    shift_fn = (shift_cv_dir / f'sub-{sub_id}' / 'func' /
                f'sub-{sub_id}_task-abstractvalue_space-T1w_desc-cvr2{smooth_label}_pe.nii.gz')

    ref = nli.load_img(str(std_fn))

    roi_masks = {}
    for roi_name, hemi in ROIS:
        try:
            roi_masks[roi_name] = sub.get_roi_mask(roi_name, hemi=hemi)
        except FileNotFoundError:
            print(f'  sub-{sub_id}: no {roi_name} mask, skipping ROI')

    for roi_name, mask_img in roi_masks.items():
        masker = NiftiMasker(mask_img=mask_img,
                             target_affine=ref.affine,
                             target_shape=ref.shape[:3]).fit()
        cvr2_std   = masker.transform(str(std_fn)).squeeze()
        cvr2_shift = masker.transform(str(shift_fn)).squeeze()

        # ── null model: leave-one-run-out, predict training mean ──────────
        sessions = sub.get_sessions()
        betas_img = sub.get_single_trial_estimates(sessions, desc='gabor',
                                                    smoothed=SMOOTHED)
        betas = pd.DataFrame(masker.transform(betas_img).astype(np.float32))

        # Build run labels matching trial order
        run_labels = []
        for ses in sessions:
            runs = sub.get_runs(ses)
            events = sub.get_events(ses, runs)
            for run in runs:
                n = len(events.loc[run].reset_index()
                        .query("event_type == 'gabor'"))
                run_labels.extend([(ses, run)] * n)
        run_labels = pd.MultiIndex.from_tuples(run_labels, names=['session', 'run'])
        betas.index = run_labels

        all_runs = sorted(set(run_labels))
        null_r2s = []
        for test_ses, test_run in all_runs:
            test_mask = (betas.index.get_level_values('session') == test_ses) & \
                        (betas.index.get_level_values('run') == test_run)
            train_mean = betas.loc[~test_mask].mean(axis=0)
            test_data = betas.loc[test_mask]
            # R² of predicting test with training mean
            pred = pd.DataFrame(
                np.tile(train_mean.values, (len(test_data), 1)),
                index=test_data.index, columns=test_data.columns)
            null_r2s.append(get_rsq(test_data, pred))
        cvr2_null = pd.concat(null_r2s, axis=1).mean(axis=1).values

        # ── filter: at least one encoding model has cvr2 > 0 ─────────────
        keep = (cvr2_std > 0) | (cvr2_shift > 0)
        n_total = len(cvr2_std)
        n_keep = keep.sum()

        data[(sub_id, roi_name)] = dict(
            cvr2_std=cvr2_std[keep],
            cvr2_shift=cvr2_shift[keep],
            cvr2_null=cvr2_null[keep],
        )

        diff = cvr2_shift[keep] - cvr2_std[keep]
        print(f'sub-{sub_id} {roi_name}: {n_keep}/{n_total} voxels (cvr2>0 in ≥1 model), '
              f'shift > std in {(diff>0).sum()}/{n_keep} ({(diff>0).sum()/n_keep*100:.1f}%), '
              f'mean diff = {diff.mean():.4f}')

## Scatter: standard vs session-shift CV R²

In [ ]:
if data:
    roi_names = sorted({roi for _, roi in data.keys()})
    fig, axes = plt.subplots(len(SUBJECTS), len(roi_names),
                              figsize=(5 * len(roi_names), 5 * len(SUBJECTS)),
                              squeeze=False, constrained_layout=True)

    for i, sub_id in enumerate(SUBJECTS):
        for j, roi_name in enumerate(roi_names):
            ax = axes[i, j]
            key = (sub_id, roi_name)
            if key not in data:
                ax.set_visible(False)
                continue

            d = data[key]
            ax.scatter(d['cvr2_std'], d['cvr2_shift'],
                       s=2, alpha=0.3, linewidths=0, c='steelblue')
            lims = [min(d['cvr2_std'].min(), d['cvr2_shift'].min()),
                    max(d['cvr2_std'].max(), d['cvr2_shift'].max())]
            ax.plot(lims, lims, 'k--', lw=1, alpha=0.5)
            ax.set_xlabel('CV R² (standard)')
            ax.set_ylabel('CV R² (session-shift)')
            ax.set_title(f'sub-{sub_id}  {roi_name}')
            ax.set_aspect('equal')

            diff = d['cvr2_shift'] - d['cvr2_std']
            ax.text(0.05, 0.95,
                    f'n={len(diff)} (cvr2>0)\n'
                    f'mean diff={diff.mean():.4f}\n'
                    f'shift > std: {(diff>0).sum()}/{len(diff)}',
                    transform=ax.transAxes, fontsize=8, va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    fig.suptitle('CV R²: standard vs session-shift (voxels with cvr2 > 0 in ≥1 model)',
                 fontsize=13)
    plt.show()

## Distribution of CV R² difference (shift - standard)

In [ ]:
if data:
    roi_names = sorted({roi for _, roi in data.keys()})
    fig, axes = plt.subplots(len(SUBJECTS), len(roi_names),
                              figsize=(5 * len(roi_names), 3.5 * len(SUBJECTS)),
                              squeeze=False, constrained_layout=True)

    for i, sub_id in enumerate(SUBJECTS):
        for j, roi_name in enumerate(roi_names):
            ax = axes[i, j]
            key = (sub_id, roi_name)
            if key not in data:
                ax.set_visible(False)
                continue

            d = data[key]
            diff = d['cvr2_shift'] - d['cvr2_std']
            ax.hist(diff, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
            ax.axvline(0, color='k', lw=1.5, ls='--')
            ax.axvline(diff.mean(), color='tomato', lw=2,
                       label=f'mean={diff.mean():.4f}')
            ax.axvline(np.median(diff), color='orange', lw=2, ls=':',
                       label=f'median={np.median(diff):.4f}')
            ax.set_xlabel('CV R² (shift) - CV R² (standard)')
            ax.set_ylabel('voxel count')
            ax.set_title(f'sub-{sub_id}  {roi_name}')
            ax.legend(fontsize=8)

    fig.suptitle('CV R² improvement from session-shift model', fontsize=13)
    sns.despine()
    plt.show()

## Summary across participants

In [ ]:
if data:
    rows = []
    for (sub_id, roi_name), d in data.items():
        for model, vals in [('null', d['cvr2_null']),
                             ('standard', d['cvr2_std']),
                             ('session-shift', d['cvr2_shift'])]:
            rows.append(dict(
                subject=sub_id, roi=roi_name, model=model,
                mean_cvr2=vals.mean(),
                median_cvr2=np.median(vals),
                pct_positive=(vals > 0).mean() * 100,
                n_voxels=len(vals),
            ))

    summary = pd.DataFrame(rows)
    print(summary.round(4).to_string(index=False))

    # Pointplot: mean CV R² by model
    if summary['subject'].nunique() >= 1:
        fig, axes = plt.subplots(1, len(ROIS), figsize=(5 * len(ROIS), 4.5),
                                  squeeze=False, constrained_layout=True)
        for ax, (roi_name, _) in zip(axes[0], ROIS):
            roi_df = summary[summary['roi'] == roi_name]
            if roi_df.empty:
                ax.set_visible(False)
                continue
            sns.stripplot(data=roi_df, x='model', y='mean_cvr2',
                          order=['null', 'standard', 'session-shift'],
                          color='0.3', size=8, jitter=0.05, ax=ax)
            # Connect dots per subject
            for sub_id in roi_df['subject'].unique():
                sub_df = roi_df[roi_df['subject'] == sub_id].set_index('model')
                xs = [0, 1, 2]
                ys = [sub_df.loc[m, 'mean_cvr2'] if m in sub_df.index else np.nan
                      for m in ['null', 'standard', 'session-shift']]
                ax.plot(xs, ys, '-', color='0.7', lw=1, zorder=0)

            ax.axhline(0, color='k', lw=1, ls='--', alpha=0.4)
            ax.set_xlabel('')
            ax.set_ylabel('Mean CV R²')
            ax.set_title(roi_name)

        fig.suptitle('Mean CV R² by model (filtered voxels)', fontsize=13)
        sns.despine()
        plt.show()

## Proportion of voxels where each model wins

In [ ]:
# ── proportion of voxels where each model has highest CV R² ──────────────
if data:
    prop_rows = []
    for (sub_id, roi_name), d in data.items():
        cvr2 = np.stack([d['cvr2_null'], d['cvr2_std'], d['cvr2_shift']], axis=1)
        winner = np.argmax(cvr2, axis=1)
        model_names = ['null', 'standard', 'session-shift']
        n = len(winner)
        for idx, model in enumerate(model_names):
            prop_rows.append(dict(
                subject=sub_id, roi=roi_name, model=model,
                proportion=(winner == idx).sum() / n,
                count=(winner == idx).sum(),
            ))

    prop_df = pd.DataFrame(prop_rows)

    roi_names = sorted(prop_df['roi'].unique())
    fig, axes = plt.subplots(1, len(roi_names),
                              figsize=(5 * len(roi_names), 4.5),
                              squeeze=False, constrained_layout=True)

    model_palette = {'null': '0.6', 'standard': 'steelblue', 'session-shift': 'tomato'}

    for ax, roi_name in zip(axes[0], roi_names):
        roi_df = prop_df[prop_df['roi'] == roi_name]
        sns.barplot(data=roi_df, x='model', y='proportion', hue='model',
                    order=['null', 'standard', 'session-shift'],
                    palette=model_palette, ax=ax, legend=False)
        # Overlay individual subjects as strips
        sns.stripplot(data=roi_df, x='model', y='proportion',
                      order=['null', 'standard', 'session-shift'],
                      color='k', size=6, alpha=0.7, jitter=0.05, ax=ax)
        ax.set_xlabel('')
        ax.set_ylabel('Proportion of voxels')
        ax.set_title(roi_name)
        ax.set_ylim(0, 1)
        ax.axhline(1/3, color='k', lw=1, ls=':', alpha=0.3, label='chance')

    fig.suptitle('Proportion of voxels where each model has highest CV R²',
                 fontsize=13)
    sns.despine()
    plt.show()

    # Print table
    pivot = (prop_df.pivot_table(index=['subject', 'roi'], columns='model',
                                 values='proportion')
             [['null', 'standard', 'session-shift']])
    print(pivot.round(3).to_string())